<a href="https://colab.research.google.com/github/sania-azhmee22/CSE470/blob/main/SANIA_part_A_CSE712.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.calibration import LabelEncoder
from tensorflow.keras.utils import to_categorical
import re
import numpy as np
from tensorflow.keras.models import Sequential
from sklearn.model_selection import train_test_split
import nltk
import tensorflow as tf
from nltk.tokenize import word_tokenize
nltk.download('punkt')
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import GradientBoostingClassifier
from tensorflow.keras.layers import Embedding , LSTM , Dense , Attention , Dropout ,Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import plot_model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving dataset.csv to dataset.csv


In [ ]:
import pandas as pd

# Load the CSV file into a DataFrame
data = pd.read_csv("dataset.csv")

In [ ]:
data.head()

,rewire_id,text,label_sexist,label_category
0,sexism2022_english-7358,"Damn, this writing was pretty chaotic",not sexist,none
1,sexism2022_english-2367,"Yeah, and apparently a bunch of misogynistic v...",not sexist,none
2,sexism2022_english-3073,How the FUCK is this woman still an MP!!!???,not sexist,none
3,sexism2022_english-14895,Understand. Know you're right. At same time I ...,not sexist,none
4,sexism2022_english-4118,Surprized they didn't stop and rape some women,not sexist,none


In [ ]:
data=data[['text','label_sexist','label_category']]

In [ ]:
label_encoder = LabelEncoder()
data['label_sexist'] = label_encoder.fit_transform(data['label_sexist'])

In [ ]:
num_classes = len(set(data['label_sexist']))
data['label_sexist'] = to_categorical(data['label_sexist'], num_classes=num_classes)

In [ ]:

label_mapping = {label: idx for idx, label in enumerate(set(data['label_category']))}
data['label_category'] = data['label_category'].map(label_mapping)


data['label_category'].fillna(-1, inplace=True)

num_classes = len(label_mapping)
data['label_sexist'] = to_categorical(data['label_category'], num_classes=num_classes)

In [ ]:
data

,text,label_sexist,label_category
0,"Damn, this writing was pretty chaotic",0.0,4
1,"Yeah, and apparently a bunch of misogynistic v...",0.0,4
2,How the FUCK is this woman still an MP!!!???,0.0,4
3,Understand. Know you're right. At same time I ...,0.0,4
4,Surprized they didn't stop and rape some women,0.0,4
...,...,...,...
13995,complexes like the 'nice chicks' that go after...,0.0,2
13996,"""GRAPHIC Germany - Muslim ""refugee"" stabbing h...",0.0,4
13997,Lol I imagine there would be simps that are li...,0.0,4
13998,"It's not, the girls I go on dates with don't k...",0.0,4


In [ ]:
def clean_text(text):
    text = text.lower()

    text = re.sub('[^a-zA-Z]', ' ', text)

    text = re.sub('\s+', ' ', text).strip()
    return text

In [ ]:
data['text'] = data['text'].apply(clean_text)

In [ ]:
data

,text,label_sexist,label_category
0,damn this writing was pretty chaotic,0.0,4
1,yeah and apparently a bunch of misogynistic vi...,0.0,4
2,how the fuck is this woman still an mp,0.0,4
3,understand know you re right at same time i kn...,0.0,4
4,surprized they didn t stop and rape some women,0.0,4
...,...,...,...
13995,complexes like the nice chicks that go after b...,0.0,2
13996,graphic germany muslim refugee stabbing his yo...,0.0,4
13997,lol i imagine there would be simps that are li...,0.0,4
13998,it s not the girls i go on dates with don t ki...,0.0,4


In [ ]:
X = data.drop(columns=['label_sexist','label_category'])
y = data.drop(columns=['text','label_category'])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
print("Training set shape:", X_train.shape, y_train.shape)
print("Testing set shape:", X_test.shape, y_test.shape)

Training set shape: (11200, 1) (11200, 1)
Testing set shape: (2800, 1) (2800, 1)


In [ ]:
tfidf_vectorizer = TfidfVectorizer()

In [ ]:
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train['text'])
X_test_tfidf = tfidf_vectorizer.transform(X_test['text'])
y_train_tf = tf.convert_to_tensor(y_train.values, dtype=tf.float32)

In [ ]:
texts_train = X_train['text'].tolist()
text_val = X_test['text'].tolist()
# Tokenize the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts_train)
X_train_sequences = tokenizer.texts_to_sequences(texts_train)
#=============================================================
tokenizer.fit_on_texts(text_val)
X_val_sequences = tokenizer.texts_to_sequences(text_val)
#=============================================================
# Pad sequences to ensure they have the same length
max_sequence_length = 70
X_train_padded = pad_sequences(X_train_sequences, maxlen=max_sequence_length)
X_val_padded = pad_sequences(X_val_sequences, maxlen=max_sequence_length)
X_train_padded_tf=tf.convert_to_tensor(X_train_padded, dtype=tf.float32)
X_val_padded_tf=tf.convert_to_tensor(X_val_padded, dtype=tf.float32)
y_val_tf=tf.convert_to_tensor(y_test, dtype=tf.float32)


In [ ]:
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import RMSprop
model = Sequential()
model.add(Embedding(input_dim=len(tokenizer.word_index) + 1, output_dim=50, input_length=max_sequence_length))
model.add(Bidirectional(LSTM(128, return_sequences=True)))
model.add(Dropout(0.5))
model.add(Bidirectional(LSTM(128)))
model.add(Dense(64, activation='relu', kernel_regularizer=l2(0.01)))
model.add(Dropout(0.3))
model.add(Dense(1, activation='sigmoid'))
learning_rate = 0.001  # Set your desired learning rate

# Instantiate RMSprop optimizer with the specified learning rate
optimizer = RMSprop(learning_rate=learning_rate)

model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.metrics import f1_score

# Oversample the minority class using SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_padded_tf, y_train_tf)

# Convert y_train_resampled to a NumPy array and ensure it's 1-dimensional
y_train_resampled_np = np.squeeze(y_train_resampled)

# Convert y_train_resampled_np to integers
y_train_resampled_np = y_train_resampled_np.astype(int)

# Calculate class weights to handle class imbalance
class_labels_resampled = np.unique(y_train_resampled_np)
class_counts_resampled = np.bincount(y_train_resampled_np)
total_samples_resampled = np.sum(class_counts_resampled)
class_weights_resampled = {label: total_samples_resampled / (len(class_labels_resampled) * count)
                           for label, count in enumerate(class_counts_resampled)}
early_stopping = tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)

# Fit the model with resampled data and class weights
history = model.fit(X_train_resampled, y_train_resampled, epochs=3, batch_size=42,
                    callbacks=[early_stopping], validation_data=(X_val_padded_tf, y_val_tf),
                    class_weight=class_weights_resampled)

# Predict on validation data
y_pred = model.predict(X_val_padded_tf)
y_pred_binary = np.round(y_pred)

# Compute F1 score
f1 = f1_score(y_val_tf, y_pred_binary)

print("F1 Score:", f1)

Epoch 1/3
521/521 [==============================] - 316s 574ms/step - loss: 0.6096 - accuracy: 0.7233 - val_loss: 0.4718 - val_accuracy: 0.8143
Epoch 2/3
521/521 [==============================] - 287s 552ms/step - loss: 0.3652 - accuracy: 0.8516 - val_loss: 0.6080 - val_accuracy: 0.7818
Epoch 3/3
88/88 [==============================] - 14s 148ms/step
F1 Score: 0.02119205298013245


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import re
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import Callback, EarlyStopping
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.regularizers import l2
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from google.colab import files
import nltk
nltk.download('punkt')

# Define F1 Score Callback
class F1ScoreCallback(Callback):
    def __init__(self, model, x_val, y_val):
        super(F1ScoreCallback, self).__init__()
        self.model = model
        self.x_val = x_val
        self.y_val = y_val

    def on_epoch_end(self, epoch, logs=None):
        y_pred = self.model.predict(self.x_val)
        y_pred_binary = (y_pred > 0.5).astype(int)
        f1 = f1_score(self.y_val.numpy().flatten(), y_pred_binary.flatten(), average='binary')
        print(f"Epoch {epoch+1}/{self.params['epochs']} | Loss: {logs['loss']:.4f} | Accuracy: {logs['accuracy']:.4f} | F1 Score: {f1:.4f}")

# Upload and read the dataset
uploaded = files.upload()
data = pd.read_csv("dataset.csv")
data.head()

# Select relevant columns and process labels
data = data[['text', 'label_sexist']]
label_encoder = LabelEncoder()
data['label_sexist'] = label_encoder.fit_transform(data['label_sexist'])
data['label_sexist'] = to_categorical(data['label_sexist'])

# Clean text data
def clean_text(text):
    text = text.lower()
    text = re.sub('[^a-zA-Z]', ' ', text)
    text = re.sub('\s+', ' ', text).strip()
    return text

data['text'] = data['text'].apply(clean_text)

# Split data into features and target
X = data['text']
y = data['label_sexist']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Tokenize text
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)
X_train_sequences = tokenizer.texts_to_sequences(X_train)
X_test_sequences = tokenizer.texts_to_sequences(X_test)

# Pad sequences
max_sequence_length = 70
X_train_padded = pad_sequences(X_train_sequences, maxlen=max_sequence_length)
X_test_padded = pad_sequences(X_test_sequences, maxlen=max_sequence_length)

# Build the model
model = Sequential()
model.add(Embedding(input_dim=len(tokenizer.word_index) + 1, output_dim=50, input_length=max_sequence_length))
model.add(Bidirectional(LSTM(128, return_sequences=True)))
model.add(Dropout(0.5))
model.add(Bidirectional(LSTM(128)))
model.add(Dense(64, activation='relu', kernel_regularizer=l2(0.01)))
model.add(Dropout(0.3))
model.add(Dense(1, activation='sigmoid'))  # Assuming binary classification

# Compile the model
optimizer = RMSprop(learning_rate=0.001)
model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

# Oversample with SMOTE
smote = SMOTE(random_state=42)
X_train_padded_resampled, y_train_resampled = smote.fit_resample(X_train_padded, y_train)

# Convert resampled data back to tensors for TensorFlow compatibility
X_train_padded_resampled = tf.convert_to_tensor(X_train_padded_resampled, dtype=tf.float32)
y_train_resampled = tf.convert_to_tensor(y_train_resampled, dtype=tf.float32)
X_test_padded = tf.convert_to_tensor(X_test_padded, dtype=tf.float32)
y_test = tf.convert_to_tensor(y_test, dtype=tf.float32)

# Initialize the F1 score callback
f1_score_callback = F1ScoreCallback(model, X_test_padded, y_test)

# Train the model with the F1 score callback
history = model.fit(
    X_train_padded_resampled,
    y_train_resampled,
    epochs=3,
    batch_size=42,
    validation_data=(X_test_padded, y_test),
    callbacks=[f1_score_callback, EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)]
)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Saving dataset.csv to dataset (1).csv
Epoch 1/3
88/88 [==============================] - 14s 138ms/step
Epoch 1/3 | Loss: 0.7561 | Accuracy: 0.5718 | F1 Score: 0.5748
406/406 [==============================] - 241s 573ms/step - loss: 0.7561 - accuracy: 0.5718 - val_loss: 0.7347 - val_accuracy: 0.5086
Epoch 2/3
88/88 [==============================] - 12s 131ms/step
Epoch 2/3 | Loss: 0.5799 | Accuracy: 0.7137 | F1 Score: 0.8757
406/406 [==============================] - 227s 559ms/step - loss: 0.5799 - accuracy: 0.7137 - val_loss: 0.5083 - val_accuracy: 0.7921
Epoch 3/3
88/88 [==============================] - 12s 134ms/step
Epoch 3/3 | Loss: 0.4895 | Accuracy: 0.7834 | F1 Score: 0.7939
406/406 [==============================] - 228s 563ms/step - loss: 0.4895 - accuracy: 0.7834 - val_loss: 0.5921 - val_accuracy: 0.7050
